Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Pathb
from scipy import stats

Diretorios:

In [2]:
# Diretórios (ALTERAÇÃO ÚNICA)
DATA_DIR = Path("/content")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

Carrega dados

In [3]:
df = pd.read_csv(DATA_DIR / "df_clean.csv")

print(f"✓ Dataset carregado: {len(df)} linhas × {len(df.columns)} colunas")
print(f"  Anos: {sorted(df['ANO'].unique())}")

✓ Dataset carregado: 3030 linhas × 20 colunas
  Anos: [np.int64(2022), np.int64(2023), np.int64(2024)]


PERGUNTA 1: ADEQUAÇÃO DO NÍVEL (IAN)

In [4]:
ian_stats = df.groupby('ANO')['IAN'].agg(['mean', 'median', 'std', 'min', 'max'])
print("Estatísticas de IAN por ano:")
print(ian_stats)
print()

# Categorizar defasagem
def categorize_defasagem(ian):
    if pd.isna(ian): return 'Não avaliado'
    if ian == 2.5: return 'Severamente defasado'
    if ian == 5.0: return 'Moderadamente defasado'
    if ian == 10.0: return 'Adequado'
    return 'Outro'

df['CATEGORIA_IAN'] = df['IAN'].apply(categorize_defasagem)
ian_categories = df.groupby(['ANO', 'CATEGORIA_IAN']).size().unstack(fill_value=0)
print("Distribuição de Defasagem por Ano:")
print(ian_categories)

Estatísticas de IAN por ano:
          mean  median       std  min   max
ANO                                        
2022  6.424419     5.0  2.389609  2.5  10.0
2023  7.243590     5.0  2.539585  2.5  10.0
2024  7.683824    10.0  2.504055  2.5  10.0

Distribuição de Defasagem por Ano:
CATEGORIA_IAN  Adequado  Moderadamente defasado  Severamente defasado
ANO                                                                  
2022                259                     573                    28
2023                462                     538                    14
2024                622                     531                     3


Houve melhora consistente do IAN médio de 2022 (6,42) para 2024 (7,68), indicando avanço na adequação educacional.
A mediana saltou para 10 em 2024, sugerindo aumento relevante no número de alunos plenamente adequados.
O número de alunos adequados cresceu fortemente (259 → 622), enquanto a defasagem severa caiu quase a zero (28 → 3).

PERGUNTA 2: DESEMPENHO ACADÊMICO (IDA)

In [5]:
ida_stats = df.groupby('ANO')['IDA'].agg(['mean', 'median', 'std', 'min', 'max'])
print("Estatísticas de IDA por ano:")
print(ida_stats)
print()

# Teste ANOVA
ida_by_year = [df[df['ANO'] == year]['IDA'].dropna().values for year in sorted(df['ANO'].unique())]
f_stat, p_value = stats.f_oneway(*ida_by_year)
print(f"Teste ANOVA para IDA: F={f_stat:.4f}, p-value={p_value:.4f}")

Estatísticas de IDA por ano:
          mean    median       std  min   max
ANO                                          
2022  6.092907  6.300000  2.046209  0.0   9.9
2023  6.663394  6.663394  1.533449  0.0  10.0
2024  6.351422  6.500000  2.036306  0.0  10.0

Teste ANOVA para IDA: F=21.5397, p-value=0.0000


O desempenho médio melhorou de 2022 (6,09) para 2023 (6,66), indicando avanço no aprendizado.
Em 2024 houve leve queda (6,35), sugerindo perda de parte do ganho anterior.
A variabilidade voltou a aumentar em 2024, indicando maior desigualdade no desempenho.
O teste ANOVA significativo (p≈0) confirma que as diferenças entre os anos são estatisticamente relevantes.

Houve melhora no desempenho até 2023, mas sem sustentação em 2024, com sinais de maior heterogeneidade entre os alunos.

PERGUNTA 3: ENGAJAMENTO (IEG) vs DESEMPENHO (IDA) e PONTO DE VIRADA (IPV)

In [6]:
corr_ieg = df[['IEG', 'IDA', 'IPV']].corr()
print("Matriz de Correlação (IEG, IDA, IPV):")
print(corr_ieg)
print()

# Teste de correlação
corr_ieg_ida, p_ieg_ida = stats.pearsonr(df['IEG'].dropna(), df['IDA'].dropna())
corr_ieg_ipv, p_ieg_ipv = stats.pearsonr(df['IEG'].dropna(), df['IPV'].dropna())

print(f"Correlação IEG vs IDA: r={corr_ieg_ida:.4f}, p-value={p_ieg_ida:.4e}")
print(f"Correlação IEG vs IPV: r={corr_ieg_ipv:.4f}, p-value={p_ieg_ipv:.4e}")

Matriz de Correlação (IEG, IDA, IPV):
          IEG       IDA       IPV
IEG  1.000000  0.389629  0.424485
IDA  0.389629  1.000000  0.556784
IPV  0.424485  0.556784  1.000000

Correlação IEG vs IDA: r=0.3896, p-value=2.0321e-110
Correlação IEG vs IPV: r=0.4245, p-value=7.8781e-133


Existe correlação positiva moderada entre engajamento e desempenho (r≈0,39), indicando que alunos mais engajados tendem a ter melhor desempenho.
O engajamento também se relaciona com o ponto de virada (r≈0,42), sugerindo que maior participação pode favorecer mudanças positivas no percurso educacional.
A relação mais forte ocorre entre desempenho e ponto de virada (r≈0,56), indicando que melhorias no aprendizado estão associadas a avanços estruturais no progresso do aluno.
Os p-values muito baixos confirmam que essas relações são estatisticamente significativas.

Engajamento contribui para o desempenho e para mudanças positivas no aprendizado, mas o principal motor do progresso educacional é o próprio desempenho acadêmico.


PERGUNTA 4: AUTOAVALIAÇÃO (IAA) vs REALIDADE (IDA e IEG)

In [7]:
corr_iaa = df[['IAA', 'IDA', 'IEG']].corr()
print("Matriz de Correlação (IAA, IDA, IEG):")
print(corr_iaa)
print()

# Desalinhamento percepção-realidade
df['DESALINHAMENTO'] = df['IAA'] - df['IDA']
desalinhamento_stats = df['DESALINHAMENTO'].describe()
print("Estatísticas de Desalinhamento (IAA - IDA):")
print(desalinhamento_stats)
print()

# Proporção de superestimação
superestimacao = (df['DESALINHAMENTO'] > 0).sum() / len(df)
print(f"Proporção de alunos que superestimam: {superestimacao:.1%}")

Matriz de Correlação (IAA, IDA, IEG):
          IAA       IDA       IEG
IAA  1.000000  0.112289  0.060183
IDA  0.112289  1.000000  0.389629
IEG  0.060183  0.389629  1.000000

Estatísticas de Desalinhamento (IAA - IDA):
count    3030.000000
mean        1.535718
std         3.011568
min        -9.900000
25%         0.300000
50%         1.900000
75%         3.252000
max         9.600000
Name: DESALINHAMENTO, dtype: float64

Proporção de alunos que superestimam: 81.1%


A autoavaliação tem correlação muito fraca com desempenho (r≈0,11) e engajamento (r≈0,06), indicando baixa aderência entre percepção e realidade.
O desalinhamento médio positivo (~1,54) mostra que os alunos tendem a se avaliar melhor do que realmente performam.
A alta dispersão indica grande variabilidade na percepção individual.
81% dos alunos superestimam seu desempenho, evidenciando viés otimista significativo.

A autoavaliação dos alunos não reflete bem o desempenho real, com predominância de superestimação.

PERGUNTA 5: ASPECTOS PSICOSSOCIAIS (IPS)

In [8]:
ips_stats = df.groupby('ANO')['IPS'].agg(['mean', 'median', 'std'])
print("Estatísticas de IPS por ano:")
print(ips_stats)
print()

corr_ips = df[['IPS', 'IDA', 'IEG']].corr()
print("Matriz de Correlação (IPS, IDA, IEG):")
print(corr_ips)

Estatísticas de IPS por ano:
          mean  median       std
ANO                             
2022  6.905000    7.50  1.070707
2023  5.119714    5.02  2.017851
2024  6.829670    7.51  1.363386

Matriz de Correlação (IPS, IDA, IEG):
          IPS       IDA       IEG
IPS  1.000000  0.019627 -0.078055
IDA  0.019627  1.000000  0.389629
IEG -0.078055  0.389629  1.000000


O IPS apresentou queda relevante em 2023, mas voltou a níveis próximos de 2022 em 2024, indicando recuperação do bem-estar psicossocial.
As correlações com desempenho (IDA) e engajamento (IEG) são muito fracas, sugerindo que fatores psicossociais têm baixa relação direta com os resultados acadêmicos.

Os aspectos psicossociais variaram ao longo do tempo, mas mostram pouca influência direta no desempenho e engajamento dos alunos.

PERGUNTA 6: ASPECTOS PSICOPEDAGÓGICOS (IPP) vs DEFASAGEM (IAN)

In [9]:
corr_ipp = df[['IPP', 'IAN']].corr()
print("Matriz de Correlação (IPP, IAN):")
print(corr_ipp)
print()

ipp_by_defasagem = df.groupby('CATEGORIA_IAN')['IPP'].agg(['mean', 'std', 'count'])
print("IPP por Categoria de Defasagem:")
print(ipp_by_defasagem)

Matriz de Correlação (IPP, IAN):
          IPP       IAN
IPP  1.000000  0.099412
IAN  0.099412  1.000000

IPP por Categoria de Defasagem:
                            mean       std  count
CATEGORIA_IAN                                    
Adequado                7.638251  0.734528   1343
Moderadamente defasado  7.492820  0.776841   1642
Severamente defasado    7.349362  0.731690     45


A correlação entre IPP e IAN é muito fraca (r≈0,10), indicando baixa relação direta entre apoio psicopedagógico e adequação ao nível.
Alunos adequados apresentam IPP ligeiramente maior, enquanto os severamente defasados têm os menores valores médios.
As diferenças são sutis, sugerindo que fatores psicopedagógicos isoladamente não explicam a defasagem.


O suporte psicopedagógico mostra associação leve com menor defasagem, mas não é um fator determinante isolado.

PERGUNTA 7: PONTO DE VIRADA (IPV) - Influências Multidimensionais

In [10]:
corr_ipv = df[['IPV', 'IPP', 'IEG', 'IDA', 'IAA', 'IPS', 'IAN']].corr()['IPV'].sort_values(ascending=False)
print("Correlações com IPV (ordenadas):")
print(corr_ipv)

Correlações com IPV (ordenadas):
IPV    1.000000
IDA    0.556784
IPP    0.492345
IEG    0.424485
IAN    0.147975
IAA    0.055587
IPS   -0.058369
Name: IPV, dtype: float64


O IPV está mais associado ao desempenho acadêmico (IDA, r≈0,56), indicando que melhorias no aprendizado são o principal fator de mudança positiva.
Também apresenta relação relevante com aspectos psicopedagógicos (IPP, r≈0,49) e engajamento (IEG, r≈0,42).
A relação com defasagem (IAN) é fraca e praticamente inexistente com autoavaliação (IAA) e aspectos psicossociais (IPS).


O ponto de virada dos alunos é principalmente impulsionado pelo desempenho e apoio pedagógico, com menor influência de fatores perceptivos ou psicossociais.

PERGUNTA 8: MULTIDIMENSIONALIDADE (INDE)

In [11]:
corr_inde = df[['INDE', 'IDA', 'IEG', 'IPV', 'IAA', 'IAN', 'IPS', 'IPP']].corr()['INDE'].sort_values(ascending=False)
print("Correlações com INDE (ordenadas):")
print(corr_inde)
print()

# Influência relativa
print("Influência relativa no INDE:")
for indicator, corr in corr_inde.items():
    if indicator != 'INDE':
        print(f"  {indicator}: {corr:.4f}")

Correlações com INDE (ordenadas):
INDE    1.000000
IDA     0.530937
IPV     0.444047
IPP     0.442711
IEG     0.384516
IAN     0.213523
IAA     0.125094
IPS     0.107900
Name: INDE, dtype: float64

Influência relativa no INDE:
  IDA: 0.5309
  IPV: 0.4440
  IPP: 0.4427
  IEG: 0.3845
  IAN: 0.2135
  IAA: 0.1251
  IPS: 0.1079


O INDE é mais influenciado pelo desempenho acadêmico (IDA, r≈0,53), sendo o principal determinante do indicador geral.
Também há influência relevante do ponto de virada (IPV) e dos aspectos psicopedagógicos (IPP), seguidos pelo engajamento (IEG).
A relação com defasagem (IAN) é moderada, enquanto autoavaliação (IAA) e aspectos psicossociais (IPS) têm impacto mais limitado.

O índice multidimensional é fortemente guiado pelo desempenho e fatores pedagógicos, com menor peso de dimensões perceptivas e psicossociais.

PERGUNTA 9: MODELO PREDITIVO

In [12]:
# Carregar metadados do modelo
import json
with open("/content/risk_model_metadata.json", 'r') as f:
    metadata = json.load(f)

print("Resultados do Modelo Preditivo:")
print(f"  Acurácia: {metadata['accuracy']:.1%}")
print(f"  Sensibilidade: {metadata['sensitivity']:.1%}")
print(f"  Especificidade: {metadata['specificity']:.1%}")
print(f"  ROC AUC: {metadata['roc_auc']:.4f}")
print(f"  F1-Score: {metadata['f1_score']:.4f}")
print(f"  CV ROC AUC: {metadata['cv_mean']:.4f} ± {metadata['cv_std']:.4f}")
print()

print("Top 5 Features Mais Importantes:")
features_df = pd.DataFrame(metadata['feature_importance'])
for idx, row in features_df.head(5).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.1%}")

Resultados do Modelo Preditivo:
  Acurácia: 78.6%
  Sensibilidade: 71.0%
  Especificidade: 83.7%
  ROC AUC: 0.8592
  F1-Score: 0.7273
  CV ROC AUC: 0.8441 ± 0.0398

Top 5 Features Mais Importantes:
  1. IAN: 19.6%
  2. IPP: 18.1%
  3. INDICADORES_MEDIA: 8.9%
  4. IPS: 8.6%
  5. IPV: 7.7%


O modelo apresenta bom desempenho geral (Acurácia ~78,6% e ROC AUC ~0,86), indicando capacidade consistente de identificar alunos em risco.
A sensibilidade de 71% mostra que o modelo captura a maior parte dos alunos em risco, enquanto a especificidade de 83,7% reduz falsos alarmes.
A validação cruzada confirma estabilidade e boa generalização dos resultados.
As variáveis mais relevantes são defasagem (IAN) e fatores psicopedagógicos (IPP), seguidas por indicadores médios, psicossociais e ponto de virada.


O modelo é confiável para prever risco educacional, sendo a defasagem prévia e o suporte pedagógico os principais determinantes do risco futuro.

 PERGUNTA 10: EFETIVIDADE DO PROGRAMA

In [13]:
# Análise de evolução por fase
if 'Fase' in df.columns:
    fase_stats = df.groupby('Fase')[['IAN', 'IDA', 'IEG', 'INDE']].agg(['mean', 'std', 'count'])
    print("Evolução de Indicadores por Fase:")
    print(fase_stats)
    print()

# Análise de melhora ao longo dos anos
print("Melhora de Indicadores (2022 → 2024):")
for ano in sorted(df['ANO'].unique()):
    ano_data = df[df['ANO'] == ano]
    print(f"  {ano}: IAN={ano_data['IAN'].mean():.2f}, IDA={ano_data['IDA'].mean():.2f}, IEG={ano_data['IEG'].mean():.2f}")

Evolução de Indicadores por Fase:
              IAN                       IDA                       IEG  \
             mean       std count      mean       std count      mean   
Fase                                                                    
0        6.802632  2.427628   190  7.140000  1.675084   190  8.088947   
1        5.742188  1.863100   192  6.464062  2.062304   192  8.522396   
1A       6.071429  2.129077    14  7.571429  1.089410    14  8.883246   
1B       7.000000  2.535463    15  7.183333  1.596499    15  8.663373   
1C       6.785714  2.486226    14  7.285714  1.999657    14  9.242063   
...           ...       ...   ...       ...       ...   ...       ...   
FASE 4   7.473404  2.706391    94  6.004255  1.247011    94  8.353191   
FASE 5   6.730769  2.574024    65  5.904615  1.389855    65  8.453846   
FASE 6   6.363636  2.261335    33  6.809091  1.184895    33  8.500000   
FASE 7   8.695652  2.244889    23  7.161918  0.727385    23  8.903806   
FASE 8  10.000000

Houve melhora consistente na adequação ao nível (IAN) ao longo do tempo, indicando avanço na redução da defasagem.
O desempenho acadêmico (IDA) melhorou até 2023, mas apresentou leve queda em 2024, sugerindo necessidade de consolidação dos ganhos.
O engajamento (IEG) cresceu em 2023, mas recuou em 2024, sinalizando possível perda de intensidade nas ações do programa.
A análise por fases mostra heterogeneidade nos resultados, indicando que o impacto do programa varia conforme o estágio do aluno.


O programa demonstra efeitos positivos na adequação educacional, mas enfrenta desafios para sustentar ganhos de desempenho e engajamento no longo prazo.

PERGUNTA 11: INSIGHTS ADICIONAIS

In [14]:
# Distribuição de risco
risco_baixo = (df['CATEGORIA_RISCO'] == 'Baixo Risco').sum() / len(df)
risco_moderado = (df['CATEGORIA_RISCO'] == 'Risco Moderado').sum() / len(df)
risco_alto = (df['CATEGORIA_RISCO'] == 'Alto Risco').sum() / len(df)
risco_muito_alto = (df['CATEGORIA_RISCO'] == 'Muito Alto Risco').sum() / len(df)

print("Distribuição de Risco:")
print(f"  Baixo Risco: {risco_baixo:.1%}")
print(f"  Risco Moderado: {risco_moderado:.1%}")
print(f"  Alto Risco: {risco_alto:.1%}")
print(f"  Muito Alto Risco: {risco_muito_alto:.1%}")
print()

# Padrão crítico
print("Padrão Crítico - Desalinhamento Percepção-Realidade:")
print(f"  Média: {df['DESALINHAMENTO_IAA_IDA'].mean():.2f}")
print(f"  Std: {df['DESALINHAMENTO_IAA_IDA'].std():.2f}")


Distribuição de Risco:
  Baixo Risco: 0.0%
  Risco Moderado: 0.0%
  Alto Risco: 44.3%
  Muito Alto Risco: 55.7%

Padrão Crítico - Desalinhamento Percepção-Realidade:
  Média: 1.54
  Std: 3.01


Resposta resumida – Insights Adicionais

A distribuição de risco permite identificar a proporção de alunos em diferentes níveis de vulnerabilidade, apoiando a priorização de intervenções educacionais.
O desalinhamento médio positivo entre autoavaliação e desempenho real indica que os alunos tendem a superestimar sua performance, com alta variabilidade entre indivíduos.


Há concentração relevante de alunos em níveis de risco que demandam atenção, além de um padrão consistente de percepção otimista em relação ao desempenho real.

RESUMO FINAL

In [15]:
print("  1. IAN: Melhora consistente 2022-2024 com forte redução da defasagem severa")
print("  2. IDA: Avanço até 2023, leve queda e maior heterogeneidade em 2024")
print("  3. IEG: Correlação moderada com IDA (~0.39) e IPV (~0.42)")
print("  4. IAA: Desalinhamento médio ~1.5 pontos; maioria dos alunos superestima desempenho")
print("  5. IPS: Variação temporal relevante, mas baixa relação direta com resultados acadêmicos")
print("  6. IPP: Associação leve com menor defasagem e forte relação com progresso educacional")
print("  7. IPV: Principalmente influenciado por IDA, seguido por IPP e IEG")
print("  8. INDE: Indicador multidimensional guiado sobretudo por desempenho e fatores pedagógicos")
print("  9. Modelo: ~78.6% acurácia, ROC AUC ~0.86, boa capacidade de identificar risco")
print("  10. Programa: Evidências de impacto positivo na adequação, com desafios de sustentação no desempenho")
print("  11. Insights: Desempenho e suporte pedagógico são motores centrais; percepção dos alunos é pouco aderente à realidade")

  1. IAN: Melhora consistente 2022-2024 com forte redução da defasagem severa
  2. IDA: Avanço até 2023, leve queda e maior heterogeneidade em 2024
  3. IEG: Correlação moderada com IDA (~0.39) e IPV (~0.42)
  4. IAA: Desalinhamento médio ~1.5 pontos; maioria dos alunos superestima desempenho
  5. IPS: Variação temporal relevante, mas baixa relação direta com resultados acadêmicos
  6. IPP: Associação leve com menor defasagem e forte relação com progresso educacional
  7. IPV: Principalmente influenciado por IDA, seguido por IPP e IEG
  8. INDE: Indicador multidimensional guiado sobretudo por desempenho e fatores pedagógicos
  9. Modelo: ~78.6% acurácia, ROC AUC ~0.86, boa capacidade de identificar risco
  10. Programa: Evidências de impacto positivo na adequação, com desafios de sustentação no desempenho
  11. Insights: Desempenho e suporte pedagógico são motores centrais; percepção dos alunos é pouco aderente à realidade
